# M8 · v1.0 · ¿Lo liberarías?

Juntás la evidencia para decidir si gente real podría usar tu sistema: **traza**, **costo**, una **protección probada con un test**. Es tu **v1.0**.

**La tarea, en cuatro pasos:** 1. **Predecí** (en `PREDICCION`) y probá con la web caída · 2. **Calculá** el costo por consulta y por mes · 3. **Escribí un test** para tu peor fallo · 4. **Decidí** go / no-go.

- Todo corre acá, en Colab: no hay archivos ni carpetas.
- Corré las celdas en orden con **Shift + Enter**. La clave: **Secretos** (la llave de la izquierda) → `GOOGLE_API_KEY`.

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
- **`buscar_web` da ERROR** (`RatelimitException`, `TimeoutException`): DuckDuckGo cortó. Para medir, `WEB = "grabada"`.
</details>


## 1 · Preparar (una vez, menos de un minuto)

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2 chromadb==1.5.9 langchain-text-splitters==1.1.3 ddgs==9.16.0
import os

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)

import time

import pandas as pd
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.5-flash-lite"
print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

## 2 · Tus documentos

Tocá **Elegir archivos** y subí tus `.md` o `.txt` (al empezar, los 5 de ejemplo del aula virtual).
Un PDF: copiá el texto a un `.txt`. Si reiniciás Colab, se suben de nuevo.

In [ ]:
from pathlib import Path

try:
    from google.colab import files
    subidos = files.upload()
except ImportError:  # fuera de Colab (solo para probar el cuaderno): una carpeta local
    carpeta = Path(os.environ.get("DOCUMENTOS", "corpus"))
    subidos = {p.name: p.read_bytes() for p in carpeta.glob("*") if p.suffix.lower() in (".md", ".txt") and not p.name.startswith("LICENSE")}
documentos = {Path(n).stem: b.decode("utf-8") for n, b in subidos.items() if n.lower().endswith((".md", ".txt"))}
print(len(documentos), "documentos:", ", ".join(documentos))

## 3 · Índice y herramientas

Los de tu v0.4.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK = 1000
OVERLAP = 150
K = 3


def indexar(chunk, overlap):
    """Parte cada documento en fragmentos y los guarda en Chroma (en memoria): busca por significado."""
    nombre = f"docs_{chunk}_{overlap}_{len(documentos)}"
    coleccion = chromadb.Client().get_or_create_collection(nombre, embedding_function=DefaultEmbeddingFunction())
    if coleccion.count():
        return coleccion
    cortador = RecursiveCharacterTextSplitter(chunk_size=chunk, chunk_overlap=overlap)
    for doc, texto in documentos.items():
        partes = cortador.split_text(texto)
        coleccion.add(ids=[f"{doc}#{i}" for i in range(len(partes))], documents=partes,
                      metadatas=[{"doc": doc, "chunk": i} for i in range(len(partes))])
    return coleccion


def buscar(coleccion, pregunta, k):
    """Los k fragmentos más parecidos a la pregunta (distancia más chica = más parecido)."""
    r = coleccion.query(query_texts=[pregunta], n_results=k)
    return [{"doc": m["doc"], "chunk": m["chunk"], "distancia": round(d, 3), "texto": t}
            for m, d, t in zip(r["metadatas"][0], r["distances"][0], r["documents"][0])]


coleccion = indexar(CHUNK, OVERLAP)
print(coleccion.count(), "fragmentos")

In [ ]:
import re
from typing import Literal

from langchain.tools import tool

WEB = "grabada"   # "real" (busca de verdad), "grabada" (siempre lo mismo: mide la elección) o "caida" (simula la caída)
RECORTE = 600     # caracteres de cada fragmento que ve el agente
GRABADA = ("- langgraph · PyPI: LangGraph is a low-level orchestration framework for building stateful agents\n"
           "- chromadb · PyPI: Chroma, the AI-native open-source embedding database\n"
           "- LangGraph es desarrollado por LangChain Inc. (github.com/langchain-ai/langgraph)")


@tool
def buscar_documentos(consulta: str) -> str:
    """Busca en la documentación del corpus (LangGraph/LangChain: persistencia, interrupts,
    subgrafos, memoria). Usar para preguntas técnicas sobre esos temas."""
    return "\n\n".join(f"[{h['doc']}] {h['texto'][:RECORTE]}" for h in buscar(coleccion, consulta, K))


@tool
def buscar_web(consulta: str) -> str:
    """Busca en internet información reciente o que no está en la documentación del corpus
    (noticias, precios, versiones nuevas, empresas, fechas)."""
    if WEB == "caida":
        return "ERROR: la búsqueda web no respondió. No inventes; decíselo al usuario."
    if WEB == "grabada":
        return GRABADA
    try:
        from ddgs import DDGS  # DuckDuckGo: gratis y sin clave
        return "\n".join(f"- {r['title']}: {r['body'][:300]}" for r in DDGS().text(consulta, max_results=3)) or "Sin resultados."
    except Exception as error:
        return f"ERROR: la búsqueda web falló ({type(error).__name__}). No inventes; decíselo al usuario."


@tool
def calcular(operacion: Literal["suma", "resta", "multiplicacion", "division", "porcentaje_cambio"],
             a: float, b: float) -> str:
    """Hace una cuenta con dos números. porcentaje_cambio = (b - a) / a * 100. Usar para cualquier cuenta."""
    resultado = {"suma": a + b, "resta": a - b, "multiplicacion": a * b, "division": a / b if b else None,
                 "porcentaje_cambio": (b - a) / a * 100 if a else None}[operacion]
    return "ERROR: división por cero." if resultado is None else f"{round(resultado, 4) + 0.0:.12g}"


def fuentes_de(mensajes):
    """Los documentos que trajo buscar_documentos (para el hit)."""
    docs = [d for m in mensajes if getattr(m, "name", None) == "buscar_documentos"
            for d in re.findall(r"^\[([^\]]+)\]", str(m.content), re.M)]
    return list(dict.fromkeys(docs))

## 4 · v1.0: traza, costo y protecciones

- **Traza:** un renglón por llamada al modelo y por herramienta, en `traza` (una lista).
- **PII:** tapa los emails de la pregunta antes de mandarla al modelo.
- **Fallo seguro:** si una herramienta dio ERROR y la respuesta no lo dice, la reemplaza.
- Precios: los oficiales, con fecha ([Gemini](https://ai.google.dev/gemini-api/docs/pricing), *Paid tier*).

In [ ]:
import uuid

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, PIIMiddleware, ToolCallLimitMiddleware, after_agent
from langchain.messages import AIMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver

SISTEMA = ("Sos un asistente de investigación. Usá buscar_documentos para LangGraph/LangChain, "
           "buscar_web para información reciente o externa y calcular para cualquier cuenta. "
           "Si una herramienta devuelve ERROR, decilo y no inventes. Respondé en español, breve.")
PRECIO_ENTRADA = 0.30   # US$ por millón (Gemini 3.5 Flash-Lite, 19/10)
PRECIO_SALIDA = 2.50
CONSULTAS_POR_DIA = 1000
DECLARA_FALLO = ("no pude", "no se puede", "falló", "fallo", "error", "no está disponible", "no respondió")
traza = []


def respuesta_segura(mensajes):
    """Si alguna herramienta dio ERROR y la respuesta no lo menciona, un texto honesto; si no, None."""
    caidas = [m.name for m in mensajes if isinstance(m, ToolMessage) and str(m.content).startswith("ERROR")]
    final = mensajes[-1] if mensajes and isinstance(mensajes[-1], AIMessage) else None
    if caidas and final and not any(f in final.text.lower() for f in DECLARA_FALLO):
        return f"No pude completar la consulta: falló {', '.join(sorted(set(caidas)))}. No invento el dato."
    return None


@after_agent
def fallo_seguro(estado: AgentState, runtime) -> None:
    texto = respuesta_segura(estado["messages"])
    if texto:
        estado["messages"][-1].content = texto


class Traza(BaseCallbackHandler):
    def on_tool_start(self, serialized, input_str, **kwargs):
        traza.append({"tipo": "herramienta", "nombre": serialized.get("name")})

    def on_llm_end(self, response, **kwargs):
        mensaje = getattr(response.generations[0][0], "message", None)
        traza.append({"tipo": "modelo", "uso": (mensaje.usage_metadata if mensaje else None) or {}})


agente = create_agent(
    init_chat_model(MODELO), tools=[buscar_documentos, buscar_web, calcular], system_prompt=SISTEMA,
    middleware=[PIIMiddleware("email", strategy="redact", apply_to_input=True),
                ModelCallLimitMiddleware(run_limit=6), ToolCallLimitMiddleware(run_limit=5), fallo_seguro],
    checkpointer=InMemorySaver())


def responder(pregunta):
    traza.append({"tipo": "consulta"})
    mensajes = agente.invoke({"messages": [{"role": "user", "content": pregunta}]},
                             {"configurable": {"thread_id": uuid.uuid4().hex}, "callbacks": [Traza()]})["messages"]
    return {"respuesta": mensajes[-1].text, "herramientas": [p["name"] for m in mensajes for p in getattr(m, "tool_calls", [])],
            "fuentes": fuentes_de(mensajes)}

## 5 · Con la web caída

In [ ]:
PREDICCION = ""  # antes de correr: con la web caída, ¿avisa o inventa? Por ejemplo: "avisa que no pudo"


In [ ]:
print("Tu predicción:", PREDICCION or "(todavía no la escribiste: hacelo antes de correr)")
WEB = "caida"
r = responder("¿Cuál es la última versión publicada de LangGraph?")
print(r["herramientas"], "\n", r["respuesta"])
WEB = "grabada"
pd.DataFrame(traza)

**Qué deberías ver:** si usó `buscar_web`, la respuesta avisa que no pudo. Si **no** la usó y dio una versión, inventó: el guardrail solo actúa ante un ERROR.

**Primer control · 10–15 min:** la traza: `"tipo": "consulta"`, cada `"tipo": "modelo"` con tokens en `uso` y cada `"tipo": "herramienta"` con su `nombre`.

## 6 · Costo por consulta y por mes

In [ ]:
traza.clear()
preguntas = ["¿Qué hace interrupt() en LangGraph?", "¿Qué empresa desarrolla LangGraph?", "Si una consulta cuesta 0.0042 USD, ¿cuánto cuestan 1500?"]
for p in preguntas:
    responder(p)
usos = [t["uso"] for t in traza if t["tipo"] == "modelo"]
entrada, salida = sum(u.get("input_tokens", 0) for u in usos), sum(u.get("output_tokens", 0) for u in usos)
costo = (entrada * PRECIO_ENTRADA + salida * PRECIO_SALIDA) / 1_000_000 / len(preguntas)
print(f"tokens: {entrada} + {salida} · por consulta: US$ {costo:.6f} · por mes: US$ {costo * CONSULTAS_POR_DIA * 30:.2f}")

## 7 · Tu peor fallo, con un test

Un test comprueba **en código**, sin gastar: armás la conversación del fallo y verificás qué hace el guardrail. Cambiá los mensajes por **tu** peor fallo.

In [ ]:
web_caida = [AIMessage("", tool_calls=[{"name": "buscar_web", "args": {"consulta": "versión"}, "id": "1"}]),
             ToolMessage("ERROR: la búsqueda web no respondió.", tool_call_id="1", name="buscar_web"),
             AIMessage("La última versión es la 9.9.")]  # inventó
assert respuesta_segura(web_caida).startswith("No pude completar la consulta")
avisa = web_caida[:2] + [AIMessage("No pude buscar en la web: falló.")]
assert respuesta_segura(avisa) is None  # si ya avisó, no la toca
print("2 tests pasan")

**Qué deberías ver:** `2 tests pasan`. Si tu fallo no viene de un ERROR de herramienta (inventó sin buscar), el `assert` falla: ese es el límite del guardrail, y también es evidencia.

**Práctica de producción** (elegí una): trazas, costos, guardrail, caché o evaluación continua. Caché en tres líneas, arriba de `responder`: `from langchain_core.caches import InMemoryCache`, `from langchain_core.globals import set_llm_cache` y `set_llm_cache(InMemoryCache())`.

## 8 · Go / no-go

En una celda de texto: «**No-go por ahora.** Suite: x/n. Costo: US$ … por consulta y por mes (precio del dd/mm). Test: pasa. Sigue fallando: … Probaría: …».

## Entregar (4 entregables hasta el dom 22/11, 23:59)

- **Guardá este notebook en tu repo:** *Archivo → Guardar una copia en GitHub* → tu repositorio → `v10.ipynb`.
- **Evidencia** (una celda de texto al final, acá mismo): un fragmento de la traza (sin claves ni datos personales), el costo con la fecha del precio, tu test y el go / no-go.

**Microdefensa:** ¿Qué te impide liberarlo hoy? ¿Qué prueba lo resolvería? Confianza del 1 al 4.